<a href="https://colab.research.google.com/github/ritan612/S-AES-OFB/blob/main/in410.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Initialization

## Initialization Vector(IV) And Input the Key

In [ ]:
import random

def initializing_IV():
  IV = random.getrandbits(16)
  # '016b' means : binary format (b), on 16 bits (16), padded with (0)
  print(f"Initialization Vector:{IV:016b}")
  return IV

def initializing_S_AES_key():
  while True:
      key = input("Enter 16-bit key: ")
      # Check length and allowed characters
      if len(key) == 16 and all(bit in '01' for bit in key):
          return key
      print("Invalid input! Please enter exactly 16 bits (only 0 and 1).")
key= initializing_S_AES_key()
# Convert to array (list of integers)
key_array = [int(bit) for bit in key]

print("Valid key:", key)
print("Key as array:", key_array)



Enter 16-bit key: 1111111100000000
Valid key: 1111111100000000
Key as array: [1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0]


## Upload the file

In [ ]:
from google.colab import drive,files

drive.mount('/content/drive') # connects the notebook to google drive so we can read/write files

uploaded = files.upload()
uploaded_path = list(uploaded.keys())[0]

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Saving Heelo World.txt to Heelo World (5).txt


## Convert the file into the Plaintext (bits array)

In [ ]:
def file_to_bits_array(file_path, chunk_size=8192):
    bits_array = []
    with open(file_path, "rb") as f: # "rb": read binary(reads raw bytes, every byte comes in as an integer 0-255)
        while chunk := f.read(chunk_size): # := (walrus operator assigns and checks in one step) read 8192 bytes and store them in chunk then check if chunk is not empty
            for byte in chunk:
                for i in range(7, -1, -1):# start from 7, count down (-1), stop befor -1 (stop on 0):these are the bit positions
                    bits_array.append((byte >> i) & 1) # ex: 01010110 for i=5 -> byte>>5 -> 01010 & 1 = 0
    return bits_array

bits = file_to_bits_array(uploaded_path)

print(bits[:64])  # first 64 bits
print(len(bits))   # total number of bits


[0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 0, 1, 0, 1, 0, 1, 1, 0, 1, 1, 0, 0, 0, 1, 1, 0, 1, 1, 0, 0, 0, 1, 1, 0, 1, 1, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 1, 0, 1, 0, 0, 0, 0, 1, 1, 0, 1, 1, 1, 1]
136


## Split The Plaintext into 16-bit Blocks

In [ ]:
from itertools import batched

def batched_with_padding(iterable, batch_size, fill_value=0):
    for batch in batched(iterable, batch_size):
      """
           "yield" returns each batch when it's ready even if the for loop is not done. That's what makes it special it doesn't act like return (because "return" returns all the batches in one data structure what makes using "return" here inefficient for the memory)
      """
      yield batch + (fill_value,) * (batch_size - len(batch))


batches = {}
""" enumerate ask the "batched_with_padding" function for an iterable and since we used yield inside this function it will returns one batch and waits until
enumerate asks for one other batch etc...
"""
for i, batch in enumerate(batched_with_padding(bits, 16), start=1):
    name = f"p{i}"
    batches[name] = batch
    print(name, "=", batch)

def bits_to_int(bits):
    return int("".join(str(b) for b in bits), 2)

p1 = (0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 0, 1, 0, 1)
p2 = (0, 1, 1, 0, 1, 1, 0, 0, 0, 1, 1, 0, 1, 1, 0, 0)
p3 = (0, 1, 1, 0, 1, 1, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0)
p4 = (0, 1, 1, 0, 1, 0, 0, 0, 0, 1, 1, 0, 1, 1, 1, 1)
p5 = (0, 1, 1, 1, 0, 1, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0)
p6 = (0, 1, 1, 0, 0, 0, 0, 1, 0, 1, 1, 1, 0, 0, 1, 0)
p7 = (0, 1, 1, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0)
p8 = (0, 1, 1, 1, 1, 0, 0, 1, 0, 1, 1, 0, 1, 1, 1, 1)
p9 = (0, 1, 1, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0)


# Simplified AES (S-AES) implementation

## SBOX and Inverse SBOX

In [ ]:
SBOX = {0x0:0x9,0x1:0x4,0x2:0xA,0x3:0xB,
        0x4:0xD,0x5:0x1,0x6:0x8,0x7:0x5,
        0x8:0x6,0x9:0x2,0xA:0x0,0xB:0x3,
        0xC:0xC,0xD:0xE,0xE:0xF,0xF:0x7}

INV_SBOX = {v:k for k,v in SBOX.items()}


## Key Generation

In [ ]:
def sub_nib(b): return (SBOX[b>>4]<<4) | SBOX[b&0xF] #t2asem el bytes 2osmen w bet tabe2 3layoun el Sbox b>>4 awal nibble b&0xF ekhir nibble
def rot_nib(b): return ((b&0xF)<<4) | (b>>4) #tbadel el nosen

def key_expansion(key): #generate sub-key
  W0 = (key>>8)&0xFF # awal 8 w byedman enno 8 bits
  W1 = key&0xFF #ekhir 8 bits
  W2 = W0 ^ 0x80 ^ sub_nib(rot_nib(W1)) #RCon[1]=80 (10000000), ^ =XOR
  W3 = W2 ^ W1
  W4 = W2 ^ 0x30 ^ sub_nib(rot_nib(W3)) #RCon[2]=30 (00110000)
  W5 = W4 ^ W3
  K0 = (W0<<8)|W1
  K1 = (W2<<8)|W3
  K2 = (W4<<8)|W5
  print(f"Key0 = {K0:016b}")
  print(f"Key1 = {K1:016b}")
  print(f"Key2 = {K2:016b}")
  return K0,K1,K2




## Encryption

In [ ]:

def add_round_key(state,key): return state ^ key #Plaintext XOR Key0

def sub_nibbles(state): # n2asem kel nibble la 7al (4 nibbles)
    return (SBOX[(state>>12)&0xF]<<12 |
            SBOX[(state>>8)&0xF]<<8 |
            SBOX[(state>>4)&0xF]<<4 |
            SBOX[state&0xF])

def shift_rows(state):
    # swap 2nd and 4th nibble
    n0=(state>>12)&0xF
    n1=(state>>8)&0xF
    n2=(state>>4)&0xF
    n3=state&0xF
    return (n0<<12)|(n3<<8)|(n2<<4)|(n1)

def mix_columns(state):
    # simplified GF(2^4) multiplication
    def mult(a,b):
        res=0
        while b: #la tsir b = 0000
            if b&1: res^=a # res = b XOR a
            a<<=1 # shift a ex.: 0100 -> 1000 / 1000 -> 10000 overflow
            if a&0x10: a^=0x13 #x⁴ + x + 1  → 0x13 (10011) ex.: 10000 XOR 10011 = 00011
            b>>=1 # shift b ex.: 0011 -> 0001
        return res&0xF
    n0=(state>>12)&0xF; n1=(state>>8)&0xF # n2asem el nibbles
    n2=(state>>4)&0xF;  n3=state&0xF
    return ((n0 ^ mult(4,n1)) << 12 |
        (mult(4,n0) ^ n1) <<  8 |
        (n2 ^ mult(4,n3)) <<  4 |
        (mult(4,n2) ^ n3))

def encrypt(plaintext,key):
    K0,K1,K2=key_expansion(key)
    print(f"Plaintext = {plaintext:016b}")
    state=add_round_key(plaintext,K0)
    print(f"After AddRoundKey0 = {state:016b}")
    state=sub_nibbles(state)
    print(f"After SubNibbles1 = {state:016b}")
    state=shift_rows(state)
    print(f"After ShiftRows1   = {state:016b}")
    state=mix_columns(state)
    print(f"After MixColumns1  = {state:016b}")
    state=add_round_key(state,K1)
    print(f"After AddRoundKey1 = {state:016b}")
    state=sub_nibbles(state)
    print(f"After SubNibbles2  = {state:016b}")
    state=shift_rows(state)
    print(f"After ShiftRows2   = {state:016b}")
    state=add_round_key(state,K2)
    print(f"Ciphertext         = {state:016b}")
    return state

In [ ]:
# Step 1: IV
iv = initializing_IV()

# make sure that key is an int
if isinstance(key, str):
    key = int(key, 2)

cipher_blocks = {}
O = iv

for i, batch in enumerate(batched_with_padding(bits, 16), start=1):
  name = f"p{i}"
  print(name, "=", bartch)

  # Step 2: Encrypt Using S-AES
    O = encrypt(O, key)
    print(f"O{i} = {O:016b}")

    # Step 3: Convert Plaintext from bits to int
    P = bits_to_int(batch)
    print(f"{name} = {P:016b}")

    # Step 4: XOR with plaintext
    C = P ^ O
    cipher_blocks[f"C{i}"] = C

    print(f"C{i} = {C:016b}\n")



## Decryption

In [ ]:
def inv_sub_nibbles(state):
    return (INV_SBOX[(state>>12)&0xF]<<12 |
            INV_SBOX[(state>>8)&0xF]<<8 |
            INV_SBOX[(state>>4)&0xF]<<4 |
            INV_SBOX[state&0xF])

def inv_shift_rows(state):
    # same as shift_rows for 2nd and 4th nibble swap
    n0=(state>>12)&0xF; n1=(state>>8)&0xF
    n2=(state>>4)&0xF;  n3=state&0xF
    return (n0<<12)|(n3<<8)|(n2<<4)|(n1)

def inv_mix_columns(state):
    def mult(a,b):
        res=0
        while b:
            if b&1: res^=a
            a<<=1
            if a&0x10: a^=0x13
            b>>=1
        return res&0xF
    n0=(state>>12)&0xF; n1=(state>>8)&0xF
    n2=(state>>4)&0xF;  n3=state&0xF
    r0=mult(9,n0)^mult(2,n1)
    r1=mult(2,n0)^mult(9,n1)
    r2=mult(9,n2)^mult(2,n3)
    r3=mult(2,n2)^mult(9,n3)
    return (r0<<12)|(r1<<8)|(r2<<4)|r3

def decrypt(ciphertext,key):
    K0,K1,K2=key_expansion(key)
    print(f"Ciphertext = {ciphertext:016b}")
    state=add_round_key(ciphertext,K2)
    print(f"After AddRoundKey2 = {state:016b}")
    state=inv_shift_rows(state)
    print(f"After InvShiftRows2 = {state:016b}")
    state=inv_sub_nibbles(state)
    print(f"After InvSubNibbles2= {state:016b}")
    state=add_round_key(state,K1)
    print(f"After AddRoundKey1  = {state:016b}")
    state=inv_mix_columns(state)
    print(f"After InvMixColumns1= {state:016b}")
    state=inv_shift_rows(state)
    print(f"After InvShiftRows1 = {state:016b}")
    state=inv_sub_nibbles(state)
    print(f"After InvSubNibbles1= {state:016b}")
    state=add_round_key(state,K0)
    print(f"Recovered Plaintext = {state:016b}")
    return state


# Implement OFB With S-AES

In [ ]:
# Step 1: IV
iv = initializing_IV()

# make sure that key is an int
if isinstance(key, str):
    key = int(key, 2)

cipher_blocks = {}
O = iv

for i, (name, bits_block) in enumerate(batches.items(), start=1):

    # Step 2: Encrypt Using S-AES
    O = encrypt(O, key)
    print(f"O{i} = {O:016b}")

    # Step 3: Convert Plaintext from bits to int
    P = bits_to_int(bits_block)
    print(f"{name} = {P:016b}")

    # Step 4: XOR with plaintext
    C = P ^ O
    cipher_blocks[f"C{i}"] = C

    print(f"C{i} = {C:016b}\n")

print("\nFinal Ciphertext:")
for name, value in cipher_blocks.items():
    print(f"{name} = {value:016b}")

binary_output = {
    "iv": format(iv, "016b"),
    "cipher": {
        name: format(value, "016b")
        for name, value in cipher_blocks.items()
    }
}

print("\n------Decryption--------")

iv = int(binary_output["iv"], 2)

cipher_blocks = {
    name: int(value, 2)
    for name, value in binary_output["cipher"].items()
}
decrypted_blocks = {}
O = iv

for i, (name, C) in enumerate(cipher_blocks.items(), start=1):
    O = encrypt(O, key)
    P = C ^ O # XOR is reversible
    decrypted_blocks[f"P{i}"] = P
    print(f"P{i} = {P:016b}\n")

Initialization Vector:0000001111100010
Plaintext = 0000001111100010
Ciphertext         = 0000001001100100
O1 = 0000001001100100
p1 = 0100100001100101
C1 = 0100101000000001

Plaintext = 0000001001100100
Ciphertext         = 0011011010000010
O2 = 0011011010000010
p2 = 0110110001101100
C2 = 0101101011101110

Plaintext = 0011011010000010
Ciphertext         = 1001111000101101
O3 = 1001111000101101
p3 = 0110111100100000
C3 = 1111000100001101

Plaintext = 1001111000101101
Ciphertext         = 0110010111100000
O4 = 0110010111100000
p4 = 0110100001101111
C4 = 0000110110001111

Plaintext = 0110010111100000
Ciphertext         = 1010111011111010
O5 = 1010111011111010
p5 = 0111011100100000
C5 = 1101100111011010

Plaintext = 1010111011111010
Ciphertext         = 0010011110110101
O6 = 0010011110110101
p6 = 0110000101110010
C6 = 0100011011000111

Plaintext = 0010011110110101
Ciphertext         = 1000010100111111
O7 = 1000010100111111
p7 = 0110010100100000
C7 = 1110000000011111

Plaintext = 10000101001

# Brute Force Attack And Cryptanalysis

In [ ]:
import time

def _key_exp(key):
    W0 = (key >> 8) & 0xFF;  W1 = key & 0xFF
    W2 = W0 ^ 0x80 ^ sub_nib(rot_nib(W1));  W3 = W2 ^ W1
    W4 = W2 ^ 0x30 ^ sub_nib(rot_nib(W3));  W5 = W4 ^ W3
    return (W0<<8)|W1, (W2<<8)|W3, (W4<<8)|W5

def _encrypt(pt, k):
    K0,K1,K2 = _key_exp(k)
    s = pt^K0
    s = sub_nibbles(s); s = shift_rows(s); s = mix_columns(s); s ^= K1
    s = sub_nibbles(s); s = shift_rows(s); s ^= K2
    return s

def _ofb_decrypt(iv_int, c_list, k):
    O = iv_int;  pt = []
    for C in c_list:
        O = _encrypt(O, k);  pt.append(C ^ O)
    return pt

def _to_text(blocks):
    raw = bytearray()
    for b in blocks: raw.append((b>>8)&0xFF); raw.append(b&0xFF)
    raw = raw.rstrip(b'\x00')
    try:    return raw.decode('utf-8')
    except: return raw.decode('latin-1')

def _printable(blocks):
    OK = set(range(0x20,0x7F))|{0x09,0x0A,0x0D,0x00}   # 0x00 = padding on last block
    for b in blocks:
        if ((b>>8)&0xFF) not in OK or (b&0xFF) not in OK: return False
    return True


_iv_int      = int(binary_output["iv"], 2)
_cipher_ints = [int(v, 2) for v in binary_output["cipher"].values()]

#  ATTACK 1 — BRUTE FORCE

print("  ATTACK 1 — BRUTE FORCE")
print(f"  Key space : 2^16 = 65536 possible keys")
print(f"  IV        : {binary_output['iv']}")
print(f"  Blocks    : {len(_cipher_ints)}")

_t0 = time.time()
_candidates = []

for _k in range(2**16):
    _pt = _ofb_decrypt(_iv_int, _cipher_ints, _k)
    if _printable(_pt):
        _candidates.append((_k, _pt))

print(f"\n  Tested  : 65536 keys in {time.time()-_t0:.2f} s")
print(f"  Found   : {len(_candidates)} candidate(s)\n")

for _k, _pt in _candidates:
    print(f"  ✓  Key      : {_k:016b}  (0x{_k:04X})")
    print(f"     Preview  : {_to_text(_pt)[:80]!r}")

    print()

#  ATTACK 2 — KNOWN-PLAINTEXT CRYPTANALYSIS
#  The attacker knows the first plaintext block P1 (e.g. file header / known text).
#  Here we take it directly from batches["p1"] to prove correctness.

_P1_known = bits_to_int(batches["p1"])   # attacker's known plaintext block

print("  ATTACK 2 — KNOWN-PLAINTEXT CRYPTANALYSIS")
print(f"  Known P1  : {_P1_known:016b}")
print(f"  C1        : {_cipher_ints[0]:016b}")
_O1_target = _cipher_ints[0] ^ _P1_known
print(f"  Target O1 : {_O1_target:016b}  (= C1 XOR P1)")
print(f"  Goal      : find k  s.t.  Encrypt(IV, k) = O1")

_t0 = time.time()
_found_key = None

for _k in range(2**16):
    if _encrypt(_iv_int, _k) == _O1_target:
        _found_key = _k
        break                     # unique solution — stop immediately

print(f"\n  Time elapsed : {time.time()-_t0:.4f} s")

if _found_key is None:
    print("  ✗  No key found — check the known plaintext value.")
else:
    print(f"  ✓  Key recovered : {_found_key:016b}  (0x{_found_key:04X})")

    _recovered = _ofb_decrypt(_iv_int, _cipher_ints, _found_key)
    print(f"\n  Decrypted blocks:")
    for i, (C, P) in enumerate(zip(_cipher_ints, _recovered), 1):
        print(f"    C{i} = {C:016b}  →  P{i} = {P:016b}")
    print(f"\n  Recovered text : {_to_text(_recovered)!r}")

# ── Final consistency check ───────────────────────────────────────────────────

print("  CONSISTENCY CHECK")
_bf_keys = [k for k,_ in _candidates]
if _found_key is not None and _found_key in _bf_keys:
    print("  ✓  Both attacks agree on the same key.\n")
else:
    print("  ✗  Attacks disagree — review validator or known P1.\n")

  ATTACK 1 — BRUTE FORCE
  Key space : 2^16 = 65536 possible keys
  IV        : 0000001111100010
  Blocks    : 9

  Tested  : 65536 keys in 4.98 s
  Found   : 1 candidate(s)

  ✓  Key      : 1111111100000000  (0xFF00)
     Preview  : 'Hello how are you'

  ATTACK 2 — KNOWN-PLAINTEXT CRYPTANALYSIS
  Known P1  : 0100100001100101
  C1        : 0100101000000001
  Target O1 : 0000001001100100  (= C1 XOR P1)
  Goal      : find k  s.t.  Encrypt(IV, k) = O1

  Time elapsed : 0.7236 s
  ✓  Key recovered : 1111111100000000  (0xFF00)

  Decrypted blocks:
    C1 = 0100101000000001  →  P1 = 0100100001100101
    C2 = 0101101011101110  →  P2 = 0110110001101100
    C3 = 1111000100001101  →  P3 = 0110111100100000
    C4 = 0000110110001111  →  P4 = 0110100001101111
    C5 = 1101100111011010  →  P5 = 0111011100100000
    C6 = 0100011011000111  →  P6 = 0110000101110010
    C7 = 1110000000011111  →  P7 = 0110010100100000
    C8 = 1111110011001000  →  P8 = 0111100101101111
    C9 = 1110010001000010  →  P9 =